# Module 1 — Planting Window (maize, GHA)
Estimates the **planting dekad** per maize pixel: cue-fusion green-up (Sentinel-2 NDRE + S1 SAR + FPAR) for the main seasons, or CHIRPS rainfall onset (25/20 mm) for the short rains, then the inception-report **5+7 false-start gate**.

**Where to start:** put the `planting_pipeline` folder on your Google Drive, run the cells top-to-bottom, and approve the Drive-mount and Earth-Engine sign-in prompts.

## Setup

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas 2>/dev/null
print('installed.')

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

In [ ]:
# --- config + GEE-native map (geemap: built-in EE Layers panel, toggle + opacity) ---
COUNTRY="Kenya"      # "Kenya" | "Ethiopia"
SEASON ="Long rains" # "Long rains" | "Short rains" | "Meher"
YEAR=2024
S1_ORBIT="ASCENDING"   # S1B gone (2022) -> ASCENDING has coverage over Kenya
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=0).geometry()
aoi_run = ee.Geometry.Rectangle([34.4,-1.2,37.8,1.2])   # fast test box; use `aoi` for whole country
import geemap
try:
    from google.colab import output; output.enable_custom_widget_manager()  # needed for interactive geemap in Colab
except Exception:
    pass
def new_map(zoom=7):
    m = geemap.Map(add_google_map=False, basemap="SATELLITE")  # keyless Google tiles; native EE layer control
    m.centerObject(aoi_run, zoom)
    return m
def ee_layer(m, image, vis, name, shown=True, opacity=1.0):
    m.addLayer(ee.Image(image), vis, name, shown, opacity)  # appears in the Layers panel (toggle + opacity)
    return m
print(f"{COUNTRY} · {SEASON} · {YEAR} · S1 {S1_ORBIT}")

## Planting-window estimation

In [ ]:
# --- planting dekad (onset) — cue-fusion green-up (main seasons) or rainfall onset (short rains) ---
from src import (utils, s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ,
                 ltn as LTN, planting_date as PD, wrsi_feedback as WR)
from run import crop_mask_image
kc, soil = utils.load_crop_coeffs()
rows={(r['country'],r['season']):r for r in utils.viable_products(utils.load_calendar('config/season_calendar.csv')) if r['crop'].lower()=='maize'}
r=rows[(COUNTRY,SEASON)]; ss,se=utils.sos_window_dekads(r['sos_detection_window']); mask=crop_mask_image(ee,COUNTRY,'maize',None)
if SEASON=='Short rains':
    pet=WR.pet_dekadal(ee,aoi_run,YEAR); ch=WR.chirps_dekadal(ee,aoi_run,YEAR)
    planting=WR.wrsi_onset(ee,ch,ss,se,pet_ic=pet).updateMask(mask).toInt16()
else:
    s2=S2.build_s2_dekadal(ee,aoi_run,YEAR); s1=S1.build_s1_dekadal(ee,aoi_run,YEAR,orbit=S1_ORBIT); fpar=FZ.add_fpar_dekadal(ee,aoi_run,YEAR)
    g=FZ.build_fused_greenness(ee,s2,s1,fpar); ltn=LTN.build_ltn_prior(ee,aoi_run,ss,se)
    sos=FZ.detect_sos(ee,g,mask,ss,se,ltn_sos=ltn,ltn_pad=2); planting=PD.sos_to_planting(ee,sos,'maize').toInt16()
print('planting dekad computed for', COUNTRY, SEASON)

In [ ]:
# 5+7 false-start gate (green-up seasons; short rains already carries the 25/20 mm rule)
if SEASON!='Short rains':
    ok=WR.dryspell_false_start(ee,aoi_run,planting,YEAR,dk_lo=ss,dk_hi=se+2); planting=planting.updateMask(ok)
print('valid maize pixels:', planting.reduceRegion(ee.Reducer.count(),aoi_run,250,maxPixels=int(1e13)).get('planting_dekad').getInfo())

In [ ]:
M=new_map()
ee_layer(M, planting.clip(aoi_run), {'min':ss,'max':se+3,'palette':['440154','3b528b','21908d','5dc863','fde725']}, f'Planting dekad — {SEASON}')
M   # geemap renders its own GEE-native Layers panel (toggle + opacity slider) — no extra layer control needed

*Higher dekad = later planting. Export with `ee.batch.Export.image.toDrive(...)`; see `run.py` for batch runs.*